# 🚲 Bicycle RTI Hackathon — One-Click Deployment

This notebook deploys the **complete** Bicycle Real-Time Intelligence solution into your workspace.

## What Gets Deployed (26 items)
| Stage | Items |
|-------|-------|
| 1. Infrastructure | 4 Lakehouses, 1 Eventhouse, 1 KQL Database |
| 2. Compute & Ingestion | 9 Notebooks, 2 Eventstreams |
| 3. Analytics | 2 Semantic Models, 1 Pipeline |
| 4. Presentation | 1 Report, 1 KQL Dashboard, 2 Data Agents, 2 Activators |

## Prerequisites
- An **empty** Fabric workspace with F64 or higher capacity
- **Contributor** or higher permissions on the workspace

## Instructions
1. Run **Cell 1** to install fabric-launcher (restarts Python session)
2. Run **Cell 2** to deploy all items
3. Run **Cell 3** to validate deployment
4. Run **Cell 4** to see post-deployment manual steps

In [ ]:
# =============================================================
# CELL 1 — Install fabric-launcher
# =============================================================
%pip install fabric-launcher --quiet
notebookutils.session.restartPython()

In [ ]:
# =============================================================
# CELL 2 — Deploy All Items (Staged)
# =============================================================
import notebookutils
from fabric_launcher import FabricLauncher

# ---- Configuration ----
REPO_OWNER = "kwamesefah-microsoft"   # <-- Update if you forked
REPO_NAME  = "RTI-Hackathon-Demo"
BRANCH     = "main"

# Initialize launcher
launcher = FabricLauncher(
    notebookutils,
    allow_non_empty_workspace=True,  # Safe for re-runs
    debug=True,
)

# Deploy in staged order (dependencies first)
launcher.download_and_deploy(
    repo_owner=REPO_OWNER,
    repo_name=REPO_NAME,
    branch=BRANCH,
    workspace_folder="workspace",
    item_type_stages=[
        # Stage 1: Infrastructure (Lakehouses + Eventhouse + KQL DB)
        ["Lakehouse", "Eventhouse", "KQLDatabase"],
        # Stage 2: Compute + Ingestion (Notebooks + Eventstreams)
        ["Notebook", "Eventstream"],
        # Stage 3: Analytics (Semantic Models + Pipeline)
        ["SemanticModel", "DataPipeline"],
        # Stage 4: Presentation (Report + Dashboard + Agents + Activators)
        ["Report", "KQLDashboard", "DataAgent", "Reflex"],
    ],
    validate_after_deployment=True,
    generate_report=True,
)

print("\n" + "="*60)
print("✅ DEPLOYMENT COMPLETE — 26 items deployed")
print("="*60)
print("\nNext: Run Cell 3 to validate, then Cell 4 for manual steps.")

In [ ]:
# =============================================================
# CELL 3 — Validate Deployment
# =============================================================
import sempy.fabric as fabric

ws_id = fabric.get_workspace_id()
print(f"Workspace: {ws_id}")

# List all items
items = fabric.list_items()
print(f"\nTotal items: {len(items)}")

# Group by type
from collections import Counter
type_counts = Counter(row['Type'] for _, row in items.iterrows())
print("\nItems by type:")
for t, c in sorted(type_counts.items()):
    print(f"  {t}: {c}")

# Check critical items
expected = [
    "bicycles_gold", "bikerental_bronze_raw", "bicycles_silver", "weather_bronze_raw",
    "bikerentaleventhouse",
    "RTIbikeRental", "RTI-WeatherDemo",
    "PL_BicycleRTI_Medallion",
    "Bicycle RTI Analytics", "Bicycle Ontology Model",
    "Bicycle Fleet Intelligence Agent", "ontology data agent",
]
names = set(items['Display Name'])
print("\nCritical item check:")
for e in expected:
    status = "✅" if e in names else "❌ MISSING"
    print(f"  {status} {e}")

## Post-Deployment Manual Steps

The automated deployment handles 26 items. The following items require
manual setup because their item types are not yet supported by fabric-cicd:

### 1. Run the Pipeline (First Load)
1. Go to **PL_BicycleRTI_Medallion** pipeline
2. Click **Run** — this processes: Silver → Gold → ML → Ontology → SM Refresh
3. Wait ~15-25 min for completion

### 2. Ontology + Graph Model (Post-Deploy Notebook)
1. Upload and run `Post_Deploy_Setup.ipynb` to create:
   - Bicycle_Ontology_Model_New (Ontology)
   - Bicycle_Ontology_Model_New_graph (GraphModel)
   - Cycling-Campaign-Agent (OperationsAgent)

### 3. Verify Eventstreams
Both eventstreams should auto-start after deployment:
- **RTIbikeRental** — Bicycle sample data → Lakehouse + Eventhouse
- **RTI-WeatherDemo** — Real-time weather → Eventhouse

If not running, open each Eventstream and click **Start**.

### 4. Activator Rules
Both Reflex/Activator items are deployed as shells. Add alert rules:
- **BicycleFleet_Activator**: Low stock alert, high demand surge
- **Cycling Campaign Activator**: Campaign trigger based on weather

See `docs/ACTIVATOR_SETUP.md` in the repo for rule definitions.